# MULTI-LAYER PERCEPTRON (MLP)

In [43]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader,TensorDataset
from sklearn.datasets import make_classification,make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

### 1. MODEL MLP KLASIFIKASI BINARY (Binary Classification)

In [44]:
X_bin,y_bin = make_classification(n_samples=1000,n_features=20,n_informative=8,n_classes=2,random_state=42)
X_train_bin,X_test_bin,y_train_bin,y_test_bin = train_test_split(X_bin,y_bin,test_size=0.2,random_state=42)

scaler_bin = StandardScaler()
X_train_bin = torch.tensor(scaler_bin.fit_transform(X_train_bin),dtype=torch.float32)
X_test_bin = torch.tensor(scaler_bin.fit_transform(X_test_bin),dtype=torch.float32)
y_train_bin = torch.tensor(y_train_bin,dtype=torch.float32).unsqueeze(1)
y_test_bin = torch.tensor(y_test_bin,dtype=torch.float32).unsqueeze(1)

In [45]:
class BinaryMLP(nn.Module):
    def __init__(self,input_dim):
        super(BinaryMLP,self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,25),
            nn.ReLU(),
            nn.Linear(25,15),
            nn.ReLU(),
            nn.Linear(15,1),
            nn.Sigmoid() # sigmoid agar output binner
        )

    def forward(self,X):
        return self.net(X)

model_mlp_binary = BinaryMLP(input_dim=20) #input sesuaikan jumlah fitur X
criterion_binary = nn.BCELoss()
optimizer_binary = optim.Adam(model_mlp_binary.parameters(),lr=0.02)

In [46]:
loader_binary = DataLoader(TensorDataset(X_train_bin,y_train_bin),batch_size=32,shuffle=True)
for epoch in range(100):
    for X_feature,y_target in loader_binary:
        optimizer_binary.zero_grad()
        loss = criterion_binary(model_mlp_binary(X_feature),y_target)
        loss.backward()
        optimizer_binary.step()

In [63]:
model_mlp_binary.eval()
with torch.no_grad():
    logits_binary = model_mlp_binary(X_test_bin)
    probs_binary = torch.sigmoid(logits_binary)  # Mengubah logits ke probabilitas (0 - 1)
    preds_binary_label = (probs_binary >= 0.5).float()
    accuracy_binary = ((logits_binary >= 0.5).float() == y_test_bin).float().mean()

    print(f"Test Accuracy (Binary): {accuracy_binary.item() * 100:.2f}%")
    print(f"Probabilitas (3 Sampel): {probs_binary[:3].flatten()}")
    print(f"Prediksi: {preds_binary_label[:3].flatten()}")
    print(f"Asli    : {y_test_bin[:3].flatten()}")

Test Accuracy (Binary): 89.00%
Probabilitas (3 Sampel): tensor([0.7311, 0.5000, 0.7311])
Prediksi: tensor([1., 1., 1.])
Asli    : tensor([1., 0., 1.])


### 2. MODEL MLP KLASIFIKASI MULTIKELAS (Multiclass Classification)

In [48]:
num_classes = 3
X_mult,y_mult = make_classification(n_samples=1000,n_features=20,n_informative=8,n_classes=num_classes,random_state=42)
X_train_mult,X_test_mult,y_train_mult,y_test_mult = train_test_split(X_mult,y_mult,test_size=0.2,random_state=42)

scaler_mult = StandardScaler()
X_train_mult = torch.tensor(scaler_mult.fit_transform(X_train_mult),dtype=torch.float32)
X_test_mult = torch.tensor(scaler_mult.fit_transform(X_test_mult),dtype=torch.float32)
y_train_mult = torch.tensor(y_train_mult,dtype=torch.long)
y_test_mult = torch.tensor(y_test_mult,dtype=torch.long)

In [49]:
class MulticlassMLP(nn.Module):
    def __init__(self,input_dim,num_classes):
        super(MulticlassMLP,self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,25),
            nn.ReLU(),
            nn.Linear(25,15),
            nn.ReLU(),
            nn.Linear(15,num_classes),
            nn.Softmax(dim=1)
        )
    def forward(self,X):
        return self.net(X)

model_mlp_multi = MulticlassMLP(input_dim=20, num_classes=num_classes)
criterion_mult = nn.CrossEntropyLoss()
optimizer_mult = optim.Adam(model_mlp_multi.parameters(), lr=0.01)

In [50]:
loader_multi = DataLoader(TensorDataset(X_train_mult,y_train_mult),batch_size=32,shuffle=True)
for epoch in range(100):
    for X_feature,y_target in loader_multi:
        optimizer_mult.zero_grad()
        loss = criterion_mult(model_mlp_multi(X_feature),y_target)
        loss.backward()
        optimizer_mult.step()

In [51]:
model_mlp_multi.eval()
with torch.no_grad():
    model_mult  = model_mlp_multi(X_test_mult)
    preds_multi = torch.argmax(model_mult,dim=1)
    probs_multi = torch.softmax(model_mult,dim=1)
    accuracy_multi = (preds_multi == y_test_mult).float().mean()

    print(f"Test Accuracy (Multiclass): {accuracy_multi.item() * 100:.2f}%")
    print(f"Probabilitas (3 Sampel Pertama):\n{probs_multi[:3].tolist()}\n")
    print(f"Prediksi: {preds_multi[:3].flatten()}")
    print(f"Asli    : {y_test_mult[:3].flatten()}")

Test Accuracy (Multiclass): 87.50%
Probabilitas (3 Sampel Pertama):
[[0.21194157004356384, 0.5761169195175171, 0.21194157004356384], [0.21194155514240265, 0.21194155514240265, 0.5761168599128723], [0.21194157004356384, 0.5761169195175171, 0.21194157004356384]]

Prediksi: tensor([1, 2, 1])
Asli    : tensor([1, 2, 1])


### 3. MODEL MLP REGRESI (Regression)

In [52]:
X_reg,y_reg = make_regression(n_samples=1000,n_features=20,noise=0.3,random_state=42)
X_train_reg,X_test_reg,y_train_reg,y_test_reg = train_test_split(X_reg,y_reg,test_size=0.2,random_state=42)

scaler_reg_X = StandardScaler()
scaler_reg_y = StandardScaler()

X_train_reg = torch.tensor(scaler_reg_X.fit_transform(X_train_reg),dtype=torch.float32)
X_test_reg = torch.tensor(scaler_reg_X.transform(X_test_reg),dtype=torch.float32)
y_train_reg = torch.tensor(scaler_reg_y.fit_transform(y_train_reg.reshape(-1,1)),dtype=torch.float32)
y_test_reg  = torch.tensor(scaler_reg_y.transform(y_test_reg.reshape(-1,1)),dtype=torch.float32)

In [53]:
class RegressionMLP(nn.Module):
    def __init__(self,input_dim):
        super(RegressionMLP,self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,35),
            nn.ReLU(),
            nn.Linear(35,15),
            nn.ReLU(),
            nn.Linear(15,1)
        )
    def forward(self,X):
        return self.net(X)

model_mlp_reg = RegressionMLP(input_dim=20)
cirterion_reg = nn.MSELoss()
optimizer_reg = optim.Adam(model_mlp_reg.parameters(),lr=0.02)

In [54]:
loader_reg = DataLoader(TensorDataset(X_train_reg,y_train_reg),batch_size=32,shuffle=True)
for epoch in range(100):
    for X_feature,y_target in loader_reg:
        optimizer_reg.zero_grad()
        loss = cirterion_reg(model_mlp_reg(X_feature),y_target)
        loss.backward()
        optimizer_reg.step()

In [55]:
model_mlp_reg.eval()
with torch.no_grad():
    preds_reg_scaled = model_mlp_reg(X_test_reg)
    mse_loss = cirterion_reg(preds_reg_scaled,y_test_reg)
    preds_reg = scaler_reg_y.inverse_transform(preds_reg_scaled.numpy())

    print(f"Test MSE Loss (Scaled): {mse_loss.item():.4f}")
    print(f"Prediksi: {preds_reg_scaled[:3].flatten()}")
    print(f"Asli    : {y_test_reg[:3].flatten()}")

Test MSE Loss (Scaled): 0.0006
Prediksi: tensor([-1.7232, -1.9690,  1.8779])
Asli    : tensor([-1.6650, -1.9438,  1.9003])
